# H3 ReVeal/test — chạy lại prefix 1.137 với Qwen3-Coder-30B-A3B-Instruct

Dùng nguyên văn prompt đã chuẩn bị cho profile context 8.192 bytes của run H3; không build lại graph/retrieval và không đổi thứ tự prefix. Model mới là `Qwen/Qwen3-Coder-30B-A3B-Instruct`; inference chạy local trên Kaggle bằng greedy decoding.

## Cách đưa prompt vào Kaggle
1. Tạo private Kaggle Dataset từ file `prompts_first1137_kaggle_dataset.zip` đi kèm notebook, hoặc upload riêng `prompts_first1137.jsonl`.
2. Attach dataset đó vào notebook. Cell tìm file này trong `/kaggle/input` tự động.
3. Bật GPU và Internet trong Notebook Settings ở lần đầu để cài thư viện/tải model.

> Qwen 30B MoE có khoảng 30.5B tham số tổng (3.3B active); 4-bit là lựa chọn mặc định để tiết kiệm VRAM. GPU Kaggle 16 GB có thể vẫn thiếu chỗ cho weights + context. Cell load dùng `device_map=auto`, nên nếu môi trường không đủ RAM/VRAM, cần accelerator có VRAM lớn hơn hoặc một quantization GGUF phù hợp.

Raw prompts là prompt đã fit/truncate theo profile 8192 byte từ run đã lưu; wrapper chat của Qwen thêm token điều khiển model nên đây là so sánh model mới trên cùng prompt text, không phải tái tạo request FPT/DeepSeek cũ theo từng byte ở wire format.

In [ ]:
from pathlib import Path
import os, json, hashlib, re, subprocess, sys, time

MODEL_ID = 'Qwen/Qwen3-Coder-30B-A3B-Instruct'
EXPECTED_N = 1137
EXPECTED_CONTEXT_BYTES = 8192
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0
SEED = 42
INPUT_BASENAME = 'prompts_first1137.jsonl'
OUTPUT_DIR = Path('/kaggle/working/qwen3coder30b_reveal_first1137_context8192')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('GPU:', subprocess.run(['bash','-lc','nvidia-smi'],capture_output=True,text=True).stdout[:4000])
print('Output:', OUTPUT_DIR)

## Cài dependencies

Chỉ cần chạy nếu image Kaggle chưa có phiên bản phù hợp. Model card yêu cầu Transformers mới hơn 4.51 cho kiến trúc `qwen3_moe`.

In [ ]:
# Internet phải bật cho pip và lần tải model đầu tiên.
%pip -q install -U 'transformers>=4.51.0' accelerate bitsandbytes safetensors tqdm


In [ ]:
# Tìm đúng input đã attach, sau đó xác minh record, thứ tự, prompt hash và byte-limit.
candidates = list(Path('/kaggle/input').rglob(INPUT_BASENAME))
if not candidates:
    raise FileNotFoundError(f'Attach Kaggle Dataset containing {INPUT_BASENAME} to this notebook.')
PROMPTS_PATH = candidates[0]
rows = [json.loads(line) for line in PROMPTS_PATH.open(encoding='utf-8') if line.strip()]
assert len(rows) == EXPECTED_N, f'Expected {EXPECTED_N} prompts, found {len(rows)}'
for i, row in enumerate(rows):
    prompt = row.get('prompt')
    assert isinstance(prompt, str) and prompt.strip(), f'Missing prompt at row {i}'
    digest = hashlib.sha256(prompt.encode('utf-8')).hexdigest()
    assert digest == row.get('prompt_sha256'), f'Prepared prompt hash mismatch at row {i}'
    trunc = row.get('prompt_truncation', {})
    assert trunc.get('context_limit') == EXPECTED_CONTEXT_BYTES, f'Context profile mismatch at row {i}'
    assert len(prompt.encode('utf-8')) <= EXPECTED_CONTEXT_BYTES, f'Prompt exceeds byte limit at row {i}'
assert len({r['record_id'] for r in rows}) == EXPECTED_N, 'Duplicate record_id in input'
print('Prompt file:', PROMPTS_PATH)
print('Verified prompts:', len(rows), '| first:', rows[0]['record_id'], '| last:', rows[-1]['record_id'])
print('First prompt bytes:', len(rows[0]['prompt'].encode('utf-8')))

## Load model 4-bit

Có thể attach model từ Kaggle Models thay cho tải trực tiếp từ Hugging Face nếu đã tạo/đưa model lên Kaggle. Khi dùng model input, đặt `MODEL_PATH` bên dưới thành thư mục model trong `/kaggle/input`; mặc định dùng model ID Hugging Face.

In [ ]:
import torch
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU chưa bật. Kaggle Settings → Accelerator → GPU rồi restart session."
    )

n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}: {props.name} | VRAM {props.total_memory / 2**30:.1f} GiB")

if n_gpus < 2:
    raise RuntimeError(
        f"Cell này cần 2 GPU; Kaggle hiện cấp {n_gpus} GPU."
    )

MODEL_PATH = os.environ.get("QWEN_MODEL_PATH", MODEL_ID)

# Lấy số layer từ config rồi chia đều lên hai T4.
cfg = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
n_layers = cfg.num_hidden_layers
split_at = n_layers // 2

device_map = {
    "model.embed_tokens": 0,
    "model.rotary_emb": 0,
    "model.norm": 1,
    "lm_head": 1,
}

for i in range(n_layers):
    device_map[f"model.layers.{i}"] = 0 if i < split_at else 1

print(f"Model layers: {n_layers}")
print(f"GPU 0: layers 0–{split_at - 1}")
print(f"GPU 1: layers {split_at}–{n_layers - 1}, plus norm/lm_head")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map=device_map,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

model.eval()

print("Loaded:", MODEL_PATH)
print("hf_device_map:", model.hf_device_map)

## Inference có checkpoint/resume

Mỗi output được flush ngay sau từng record. Nếu Kaggle ngắt phiên, chạy lại cell sẽ giữ raw response đã có khi `record_id` và hash prompt trùng khớp. Mỗi record dùng `temperature=0`/greedy; parser lưu nhãn `0/1` khi dòng cuối không rỗng đúng một ký tự binary, còn lại đánh dấu invalid và giữ nguyên raw text.

In [ ]:
import re, json, time, hashlib
from tqdm.auto import tqdm
RAW_PATH = OUTPUT_DIR / 'raw_responses.jsonl'
PRED_PATH = OUTPUT_DIR / 'predictions.jsonl'

def load_done(path):
    done = {}
    if path.exists():
        with path.open(encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    done[row['record_id']] = row
    return done

raw_done = load_done(RAW_PATH)
valid_done = load_done(PRED_PATH)
bad = [rid for rid, r in raw_done.items() if r.get('prompt_sha256') != next((x['prompt_sha256'] for x in rows if x['record_id']==rid), None)]
if bad:
    raise ValueError(f'Checkpoint prompt hash mismatch for {len(bad)} records; move old output before restarting.')
print('Already saved:', len(raw_done), '| remaining:', EXPECTED_N-len(raw_done))

def parse_binary(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines and lines[-1] in {'0','1'}:
        return int(lines[-1]), 'valid'
    return None, 'invalid_output'

raw_f = RAW_PATH.open('a', encoding='utf-8', buffering=1)
pred_f = PRED_PATH.open('a', encoding='utf-8', buffering=1)
try:
    for row in tqdm(rows, total=len(rows)):
        rid = row['record_id']
        if rid in raw_done:
            continue
        messages = [{'role': 'user', 'content': row['prompt']}]
        chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(chat_text, return_tensors='pt')
        input_device = model.get_input_embeddings().weight.device
        inputs = {k: v.to(input_device) for k, v in inputs.items()}
        started = time.time()
        try:
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True)
            new_tokens = generated[0, inputs['input_ids'].shape[1]:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            label, status = parse_binary(text)
            item = {'record_id': rid, 'dataset': row.get('dataset','reveal'), 'split': row.get('split','test'),
                    'prompt_sha256': row['prompt_sha256'], 'model': MODEL_ID, 'context_limit_bytes': EXPECTED_CONTEXT_BYTES,
                    'raw_text': text, 'prediction': label, 'status': status, 'elapsed_seconds': round(time.time()-started, 3)}
            raw_f.write(json.dumps(item, ensure_ascii=False) + '\n')
            raw_done[rid] = item
            if label is not None:
                pred_f.write(json.dumps({'record_id':rid, 'prediction':label, 'prompt_sha256':row['prompt_sha256'], 'model':MODEL_ID}, ensure_ascii=False) + '\n')
        except torch.cuda.OutOfMemoryError as e:
            torch.cuda.empty_cache()
            item = {'record_id':rid, 'dataset':'reveal', 'split':'test', 'prompt_sha256':row['prompt_sha256'],
                    'model':MODEL_ID, 'context_limit_bytes':EXPECTED_CONTEXT_BYTES, 'raw_text':'', 'prediction':None,
                    'status':'cuda_oom', 'error':str(e)[:1000], 'elapsed_seconds':round(time.time()-started,3)}
            raw_f.write(json.dumps(item, ensure_ascii=False) + '\n'); raw_done[rid] = item
            print('CUDA OOM at', rid, '- stop here; reduce memory use / use larger GPU, then remove this row from raw file to retry.')
            break
finally:
    raw_f.close(); pred_f.close()

statuses = {}
for r in raw_done.values(): statuses[r.get('status','unknown')] = statuses.get(r.get('status','unknown'),0)+1
print('Saved raw records:', len(raw_done), '| status counts:', statuses)
print('Raw:', RAW_PATH)
print('Valid predictions:', PRED_PATH)

## Tải output

Raw JSONL giữ nguyên cả output không hợp lệ/OOM; `predictions.jsonl` chỉ chứa nhãn hợp lệ. Tải các file từ Output pane hoặc nén để tải một lần.

In [ ]:
import zipfile
zip_path = Path('/kaggle/working/qwen3coder30b_reveal_first1137_context8192_outputs.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob('*.jsonl'):
        z.write(p, p.name)
print('Download:', zip_path, '| bytes:', zip_path.stat().st_size)